In [5]:
import pandas as pd
from fpdf import FPDF
from fpdf.enums import XPos, YPos
import os
from openpyxl import load_workbook
from datetime import datetime

# --- 1. CONFIGURAÇÕES DE CAMINHOS ---
ARQUIVO_EXCEL = r'C:\Users\Audi-02\OneDrive - Universidade de Vassouras (1)\Auditoria Interna FUSVE\AUDITORIA DE DIAGNÓSTICO FUSVE.xlsx'
PASTA_PDFS = r'C:\Users\Audi-02\OneDrive - Universidade de Vassouras (1)\Auditoria Interna FUSVE\Relatórios pdfs'
CAMINHO_LOGO = r"C:\Users\Audi-02\OneDrive - Universidade de Vassouras (1)\Auditoria Interna FUSVE\logo_fusve.png" # A logo deve estar na mesma pasta do script
CAMINHO_LOGO2 = r"C:\Users\Audi-02\OneDrive - Universidade de Vassouras (1)\Auditoria Interna FUSVE\logo_auditoria.png"
NOME_ABA = '1 DADOS DO PROC'

if not os.path.exists(PASTA_PDFS):
    os.makedirs(PASTA_PDFS)

class PDF(FPDF):
    def header(self):
        # 1. Configuração das Logos
        tamanho_logo = 25

        if os.path.exists(CAMINHO_LOGO):
            self.image(CAMINHO_LOGO, 15, 10, tamanho_logo)
        
        if os.path.exists(CAMINHO_LOGO2):
            self.image(CAMINHO_LOGO2, 15, 20, tamanho_logo)
        
        # 2. Título 
        self.set_y(12)
        self.set_x(32)
        self.set_font("helvetica", "B", 14)
        self.cell(0, 10, "RELATÓRIO DE VALIDAÇÃO DO PROCESSO", border=False, align="C", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # 2.1 Subtítulo
        self.set_x(32)
        self.set_font('helvetica', "", 10)
        self.cell(0, 5, "Diagnóstico de Auditoria Interna - FUSVE", border=False, align="C", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        
        # 3. Linha decorativa abaixo do título
        self.set_y(45)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(10)

    def footer(self):
        self.set_y(-15)
        self.set_x(170)
        self.set_font("helvetica", "I", 8)
        self.cell(0, 10, f"Página {self.page_no()}", align="C")

def gerar_relatorio(id_proc, dados_grupo):
    pdf = PDF()
    pdf.add_page()
    primeira_linha = dados_grupo.iloc[0]

    # --- SEÇÃO 1: CABEÇALHO E INFORMAÇÕES GERAIS ---
    pdf.set_fill_color(240, 240, 240)
    pdf.set_font("helvetica", "B", 10)
    
    # Grid de informações gerais
    pdf.cell(0, 8, f"ID DO PROCESSO: {id_proc}", new_x=XPos.LMARGIN, new_y=YPos.NEXT, fill=True)
    pdf.cell(0, 8, f"ÁREA: {primeira_linha['AREA']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    # PROCESSO com quebra automática de linha (usa multi_cell para wrapping)
    x_proc = pdf.get_x()
    y_proc = pdf.get_y()
    pdf.set_font("helvetica", "B", 10)
    pdf.cell(30, 8, "PROCESSO:", border=False)
    pdf.set_xy(x_proc + 30, y_proc)
    pdf.set_font("helvetica", "", 10)
    pdf.multi_cell(0, 6, str(primeira_linha['PROCESSO']), border=0, align="L")
    
    pdf.ln(2)
    pdf.set_font("helvetica", "B", 9)
    pdf.cell(0, 6, "OBJETIVO DO PROCESSO:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("helvetica", "", 9)
    pdf.multi_cell(0, 6, str(primeira_linha['OBJETIVO']))

    pdf.ln(2)
    pdf.set_font("helvetica", "B", 9)
    pdf.cell(0, 6, "DESCRIÇÃO DETALHADA:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("helvetica", "", 9)
    pdf.multi_cell(0, 6, str(primeira_linha['DESCRIÇÃO DO PROCESSO']))

    # Detalhes de execução em colunas
    pdf.ln(2)
    pdf.set_font("helvetica", "B", 9)
    pdf.cell(95, 6, f"QUEM EXECUTA: {primeira_linha['QUEM EXECUTA?']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.cell(95, 6, f"PRODUTO: {primeira_linha['PRODUTO DO PROCESSO']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    
    pdf.cell(95, 6, f"ETAPA INICIAL: {primeira_linha['ETAPA INICIAL']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.cell(95, 6, f"ETAPA FINAL: {primeira_linha['ETAPA FINAL']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    
    pdf.ln(5)

    # --- SEÇÃO 2: TABELA DE RISCOS ---
    # Cabeçalho da Tabela
    def draw_table_header():
        pdf.set_fill_color(200, 220, 255)
        pdf.set_font('helvetica', "B", 8)
        # Larguras: Risco(50), Fator(40), Melhoria(40), Imp(20), Prob(20), Bruto(20) = Total 190
        pdf.cell(50, 10, "Descrição do Risco", border=1, fill=True, align="C")
        pdf.cell(40, 10, "Fator de Risco", border=1, fill=True, align="C")
        pdf.cell(40, 10, "O que Melhorar?", border=1, fill=True, align="C")
        pdf.cell(20, 10, "Imp.", border=1, fill=True, align="C")
        pdf.cell(20, 10, "Prob.", border=1, fill=True, align="C")
        pdf.cell(20, 10, "Risco Bruto", border=1, fill=True, align="C", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font('helvetica', "", 8)

    draw_table_header()
    
    def wrap_text_lines(pdf_obj, text, width):
        # Preserva quebras de parágrafo e faz wrap por largura, quebrando palavras muito longas
        paragraphs = str(text).splitlines() or ['']
        out_lines = []
        for para in paragraphs:
            words = para.split()
            if not words:
                out_lines.append('')
                continue
            cur = ''
            for w in words:
                test = (cur + ' ' + w).strip()
                if pdf_obj.get_string_width(test) <= width:
                    cur = test
                else:
                    if cur:
                        out_lines.append(cur)
                    part = ''
                    for ch in w:
                        if pdf_obj.get_string_width(part + ch) <= width:
                            part += ch
                        else:
                            if part:
                                out_lines.append(part)
                            part = ch
                    cur = part
            if cur:
                out_lines.append(cur)
        return out_lines

    for _, linha in dados_grupo.iterrows():
        x_start = pdf.get_x()
        y_start = pdf.get_y()
        line_h = 5
        padding = 2
        widths = [50, 40, 40]
        texts = [str(linha['RISCO']), str(linha['FATOR DE RISCO']), str(linha['O QUE PODERIA MELHORAR?'])]

        # Gera linhas embaladas para cada coluna (respeita newlines)
        wrapped = [wrap_text_lines(pdf, t, w - 2*padding) for w, t in zip(widths, texts)]
        # Garantir pelo menos uma linha por coluna
        wrapped = [[ln for ln in col] if col else [''] for col in wrapped]

        max_lines = max(len(col) for col in wrapped)
        altura_linha = max_lines * line_h

        # quebra de página se a linha não couber
        if y_start + altura_linha > (pdf.h - pdf.b_margin):
            pdf.add_page()
            x_start = pdf.get_x()
            y_start = pdf.get_y()

        # desenha cada coluna com borda uniforme e escreve linha-a-linha (evita multi_cell que avança cursor)
        for i, (w, lines_list) in enumerate(zip(widths, wrapped)):
            x = x_start + sum(widths[:i])
            pdf.set_xy(x, y_start)
            pdf.rect(x, y_start, w, altura_linha)
            for j in range(max_lines):
                text_line = lines_list[j] if j < len(lines_list) else ''
                pdf.set_xy(x + padding, y_start + j*line_h + (padding/2))
                pdf.cell(w - 2*padding, line_h, text_line, border=0)

        # colunas numéricas à direita com altura uniforme (mantém alinhamento vertical)
        pdf.set_xy(x_start + sum(widths), y_start)
        pdf.cell(20, altura_linha, str(linha['IMPACTO']), border=1, align="C")
        pdf.cell(20, altura_linha, str(linha['PROBABILIDADE']), border=1, align="C")
        pdf.cell(20, altura_linha, str(int(linha['RISCO BRUTO'])), border=1, align="C")

        # posiciona o cursor exatamente abaixo da linha desenhada
        pdf.set_xy(x_start, y_start + altura_linha)

    # --- SEÇÃO 3: ASSINATURAS ---
    pdf.ln(20)
    if pdf.get_y() > 240: pdf.add_page()
    
    y_assinatura = pdf.get_y() + 10
    pdf.line(20, y_assinatura, 90, y_assinatura)
    pdf.line(110, y_assinatura, 180, y_assinatura)
    pdf.set_y(y_assinatura + 2)
    pdf.set_font("helvetica", "B", 8)

    # Primeira linha das assinaturas para os cargos
    pdf.cell(90, 5, "Gerência", align="C")
    pdf.cell(90, 5, "Superintendência", align="C", new_x=XPos.LMARGIN, new_y=YPos.NEXT)

    # Segunda linha das assinaturas para a área
    pdf.set_font("helvetica", "", 6)
    nome_area = str(primeira_linha['AREA'])
    pdf.cell(90, 5, nome_area, align="C")

    # --- SEÇÃO 4: LOCAL E DATA ---
    pdf.ln(15)
    # Configura o idioma da data (Opcional, mas garante o formato em português)
    # Pegamos a data atual e formatamos
    data_hoje = datetime.now()
    meses = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", 
             "julho", "agosto", "setembro", "outubro", "novembro", "dezembro"]
    
    data_formatada = f"Vassouras, {data_hoje.day} de {meses[data_hoje.month - 1]} de {data_hoje.year}."
    
    # Escrevendo no PDF
    pdf.set_font("helvetica", "", 10)
    pdf.set_x(20) # Alinhado com o início da linha de assinatura (margem esquerda)
    pdf.cell(0, 10, data_formatada, new_x=XPos.LMARGIN, new_y=YPos.NEXT)


    # Salvar
    nome_clean = str(id_proc).replace(".", "_")
    pdf.output(os.path.join(PASTA_PDFS, f"Relatorio_{nome_clean}.pdf"))

def rodar_automacao():
    try:
        df = pd.read_excel(ARQUIVO_EXCEL, sheet_name=NOME_ABA, header=6)
        df.columns = df.columns.str.strip()

        col_status = "RELATÓRIO GERADO? Sim/Não"
        # normaliza espaços e maiúsculas para evitar interpretações erradas
        df[col_status] = df[col_status].astype(str).str.strip().str.upper()
        # pendentes são todas as linhas que **não** estão marcadas como "SIM"
        df_pendente = df[df[col_status] != "SIM"]

        if df_pendente.empty:
            print(">>> Tudo em dia! Nenhum relatório novo para gerar.")
            return
        
        linhas_para_atualizar = []
        for id_proc, grupo in df_pendente.groupby('Nº PC'):
            print(f"Gerando relatório detalhado para o processo {id_proc}...")
            gerar_relatorio(id_proc, grupo)
            for idx in grupo.index:
                linhas_para_atualizar.append(idx + 8)

        # se nenhum índice foi adicionado, significa que não havia grupos válidos
        if not linhas_para_atualizar:
            print(">>> Tudo em dia! Nenhum relatório novo para gerar.")
            return

        print("Atualizando Excel...")
        wb = load_workbook(ARQUIVO_EXCEL)
        ws = wb[NOME_ABA]
        col_idx = df.columns.get_loc(col_status) + 1

        for linha_num in linhas_para_atualizar:
            ws.cell(row=linha_num, column=col_idx).value = "SIM"

        wb.save(ARQUIVO_EXCEL)
        print(f">>> Sucesso! Relatórios gerados com todas as novas informações.")

    except Exception as e:
        print(f"ERRO: {e}")

if __name__ == "__main__":
    rodar_automacao()

Gerando relatório detalhado para o processo 1.3...
Atualizando Excel...
>>> Sucesso! Relatórios gerados com todas as novas informações.
